# puxar_visualizar_falhas — Google Colab

Puxa do Google Drive as imagens que **falharam** no `makePredict_fromTIF_sortedByQuantity_colab.ipynb`
(TIFs de entrada sem `_pred.tif` correspondente) e faz o diagnóstico de integridade + visualização.

## Contexto das falhas
As falhas observadas no log do predict **não são do modelo**, e sim de **integridade do arquivo
no Drive** (download parcial vindo do GEE). Dois tipos:

- **`corrompido (header inválido)`** — `not recognized as being in a supported file format`.
- **`truncado (download incompleto)`** — `TIFFReadEncodedTile() failed ... got N bytes, expected M`.

Arquivos nesses dois estados **precisam ser re-exportados do GEE** — não há conserto só puxando de novo.

---
**Drive montado em `/content/drive`.**

In [ ]:
# ── Célula 1: Montar Drive e instalar dependências ───────────────────────────
from google.colab import drive
drive.mount('/content/drive')

!pip install -q rasterio matplotlib

In [ ]:
# ── Célula 2: PARÂMETROS ─────────────────────────────────────────────────────
INPUT_DIR   = "/content/drive/MyDrive/DS_FV_TIFs_scaled/PREDICT_V2"          # TIFs de entrada
OUTPUT_DIR  = "/content/drive/MyDrive/DL_fotovoltaica/tif_classificadas_2025" # onde ficam os *_pred.tif
FALHAS_DIR  = "/content/falhas_predict"   # pasta LOCAL do Colab p/ puxar cópias e inspecionar
DIAG_CSV    = "diagnostico_falhas.csv"    # salvo dentro de FALHAS_DIR

# Deixe vazio p/ detectar automaticamente (TIFs sem _pred). Ou liste nomes manualmente:
NOMES_MANUAIS = []
# Ex.: NOMES_MANUAIS = ["00000000000000000789_2025.tif", "0000000000000000085b_2025.tif"]

RGB_BANDS = (3, 2, 1)   # índices 1-based das bandas para composição RGB (R, G, B)

print("Parâmetros carregados.")
print(f"  INPUT_DIR : {INPUT_DIR}")
print(f"  OUTPUT_DIR: {OUTPUT_DIR}")
print(f"  FALHAS_DIR: {FALHAS_DIR}")

In [ ]:
# ── Célula 3: Imports ────────────────────────────────────────────────────────
import csv, shutil
from pathlib import Path
import numpy as np
import rasterio
import rasterio.errors
import matplotlib.pyplot as plt

input_dir  = Path(INPUT_DIR)
output_dir = Path(OUTPUT_DIR)
falhas_dir = Path(FALHAS_DIR)
falhas_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Célula 4: Detectar imagens faltantes (sem _pred.tif) ─────────────────────
def detectar_faltantes(input_dir: Path, output_dir: Path):
    """TIFs em input_dir que não possuem '{stem}_pred.tif' em output_dir."""
    faltantes = []
    for tif in sorted(input_dir.glob('*.tif')):
        if not (output_dir / f'{tif.stem}_pred.tif').exists():
            faltantes.append(tif)
    return faltantes

if NOMES_MANUAIS:
    alvo = [input_dir / n for n in NOMES_MANUAIS]
else:
    alvo = detectar_faltantes(input_dir, output_dir)

print(f'{len(alvo)} imagem(ns) sem predict:')
for p in alvo:
    print('  ', p.name)

In [ ]:
# ── Célula 5: Diagnóstico de integridade ─────────────────────────────────────
def diagnosticar(tif_path: Path) -> dict:
    info = {'arquivo': tif_path.name, 'existe': tif_path.exists(),
            'bytes': tif_path.stat().st_size if tif_path.exists() else 0,
            'H': None, 'W': None, 'bandas': None, 'status': '', 'erro': ''}
    if not info['existe']:
        info['status'] = 'ausente'
        return info
    try:
        with rasterio.open(tif_path) as src:
            info['H'], info['W'], info['bandas'] = src.height, src.width, src.count
            _ = src.read()   # força leitura de TODOS os tiles (pega truncamento)
        info['status'] = 'OK'
    except rasterio.errors.RasterioIOError as e:
        msg = str(e)
        if 'not recognized' in msg:
            info['status'] = 'corrompido (header inválido)'
        elif 'Read failed' in msg or 'TIFFReadEncodedTile' in msg:
            info['status'] = 'truncado (download incompleto)'
        else:
            info['status'] = 'erro de leitura'
        info['erro'] = msg
    except Exception as e:
        info['status'] = 'erro'
        info['erro'] = str(e)
    return info

diags = [diagnosticar(p) for p in alvo]

csv_out = falhas_dir / DIAG_CSV
with open(csv_out, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['arquivo', 'existe', 'bytes', 'H', 'W', 'bandas', 'status', 'erro'])
    w.writeheader()
    for d in diags:
        w.writerow(d)

print(f'Diagnóstico salvo em: {csv_out}\n')
print(f"{'arquivo':38s} {'bytes':>13s}  status")
print('-' * 80)
for d in diags:
    print(f"  {d['arquivo']:36s} {d['bytes']:>13,}  {d['status']}")

In [ ]:
# ── Célula 6: Puxar (copiar) do Drive para pasta local ───────────────────────
copiadas = []
for p in alvo:
    if not p.exists():
        print('ausente no Drive:', p.name)
        continue
    try:
        dest = falhas_dir / p.name
        shutil.copy2(p, dest)
        copiadas.append(dest)
        print('copiado ->', dest)
    except Exception as e:
        print('falha ao copiar', p.name, ':', e)

print(f'\n{len(copiadas)} arquivo(s) copiado(s) para {falhas_dir}')

In [ ]:
# ── Célula 7: Visualizar ─────────────────────────────────────────────────────
def ler_rgb(tif_path: Path, rgb_bands=RGB_BANDS) -> np.ndarray:
    with rasterio.open(tif_path) as src:
        bands = [src.read(b) for b in rgb_bands]
    rgb = np.stack(bands, axis=-1).astype(np.float32)
    for i in range(rgb.shape[-1]):        # normalização por percentis (2–98) p/ visualizar
        ch = rgb[..., i]
        lo, hi = np.percentile(ch, (2, 98))
        rgb[..., i] = np.clip((ch - lo) / (hi - lo + 1e-6), 0, 1)
    return rgb

for d in diags:
    p = falhas_dir / d['arquivo']
    if d['status'] == 'OK' and p.exists():
        try:
            rgb = ler_rgb(p)
            plt.figure(figsize=(6, 6))
            plt.imshow(rgb)
            plt.title(d['arquivo'])
            plt.axis('off')
            plt.show()
        except Exception as e:
            print('não plotou', d['arquivo'], ':', e)
    else:
        print(f"⚠ {d['arquivo']}: {d['status']} — RE-EXPORTAR do GEE (puxar de novo não conserta).")

In [ ]:
# ── Célula 8: Apagar do Drive as imagens com falha (para re-exportar do GEE) ──
# Apaga em INPUT_DIR (Drive) os TIFs com status corrompido/truncado, para que
# possam ser re-exportados do GEE. Ação IRREVERSÍVEL → protegida por flag.
#
#   CONFIRMAR_APAGAR = False  → apenas SIMULA (mostra o que apagaria)
#   CONFIRMAR_APAGAR = True   → apaga de fato
CONFIRMAR_APAGAR = False

# Só apaga esses status (nunca apaga um arquivo 'OK'):
STATUS_APAGAVEIS = {'corrompido (header inválido)', 'truncado (download incompleto)', 'erro de leitura'}

apagaveis = [d for d in diags if d['status'] in STATUS_APAGAVEIS]
print(f'{len(apagaveis)} arquivo(s) com falha marcados para apagar em {INPUT_DIR}\n')

apagados = 0
for d in apagaveis:
    origem = input_dir / d['arquivo']          # arquivo no Drive
    if not origem.exists():
        print(f"  ausente (já não está no Drive): {d['arquivo']}")
        continue
    if CONFIRMAR_APAGAR:
        try:
            origem.unlink()
            apagados += 1
            print(f"  APAGADO: {d['arquivo']}  ({d['status']})")
        except Exception as e:
            print(f"  falha ao apagar {d['arquivo']}: {e}")
    else:
        print(f"  [SIMULAÇÃO] apagaria: {d['arquivo']}  ({d['status']})")

if CONFIRMAR_APAGAR:
    print(f'\n{apagados} arquivo(s) apagado(s) do Drive. Re-exporte-os do GEE.')
else:
    print('\nSIMULAÇÃO — nada foi apagado. Defina CONFIRMAR_APAGAR = True para apagar de fato.')